## Домашнее задание к занятию "A/B-тесты"

### Описание задачи

![banner](https://storage.googleapis.com/kaggle-datasets-images/635/1204/126be74882028aac7241553cef0e27a7/dataset-original.jpg)

Покемоны - это маленькие существа, которые сражаются друг с другом на соревнованиях. Все покемоны имеют разные характеристики (сила атаки, защиты и т. д.) И относятся к одному или двум так называемым классам (вода, огонь и т. д.).
Профессор Оук является изобретателем Pokedex - портативного устройства, которое хранит информацию обо всех существующих покемонах. Как его ведущий специалист по данным, Вы только что получили от него запрос с просьбой осуществить аналитику данных на всех устройствах Pokedex.

### Описание набора данных
Профессор Оук скопировал все содержимое в память одного устройства Pokedex, в результате чего получился набор данных, с которым Вы будете работать в этой задаче. В этом файле каждая строка представляет характеристики одного покемона:

* `pid`: Numeric - ID покемона
* `HP`: Numeric - Очки здоровья
* `Attack`: Numeric - Сила обычной атаки
* `Defense`: Numeric - Сила обычной защиты
* `Sp. Atk`: Numeric - Сила специальной атаки
* `Sp. Def`: Numeric - Сила специальной защиты
* `Speed`: Numeric - Скорость движений
* `Legendary`: Boolean - «True», если покемон редкий
* `Class 1`: Categorical - Класс покемона
* `Class 2`: Categorical - Класс покемона

In [ ]:
import warnings
# Отключение предупреждений (warnings)
warnings.filterwarnings("ignore")

import pandas as pd

import scipy.stats as st
from scipy.stats import ttest_ind
from scipy.stats import f_oneway, shapiro

pokemon = pd.read_csv('https://raw.githubusercontent.com/a-milenkin/datasets_for_t-tests/main/pokemon.csv', on_bad_lines='skip')  # Откроем датасет
pokemon.head()

# Обратите внимание, что у покемона может быть один или два класса.
# Если у покемона два класса, считается, что они имеют одинаковую значимость.

,pid,Name,Class 1,Class 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Legendary
0,1,Bulbasaur,Grass,Poison,45,49,49,65,65,45,False
1,2,Ivysaur,Grass,Poison,60,62,63,80,80,60,False
2,3,Venusaur,Grass,Poison,80,82,83,100,100,80,False
3,4,Mega Venusaur,Grass,Poison,80,100,123,122,120,80,False
4,5,Charmander,Fire,NaN,39,52,43,60,50,65,False


In [ ]:
pokemon.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   pid        800 non-null    int64 
 1   Name       799 non-null    object
 2   Class 1    800 non-null    object
 3   Class 2    414 non-null    object
 4   HP         800 non-null    int64 
 5   Attack     800 non-null    int64 
 6   Defense    800 non-null    int64 
 7   Sp. Atk    800 non-null    int64 
 8   Sp. Def    800 non-null    int64 
 9   Speed      800 non-null    int64 
 10  Legendary  800 non-null    bool  
dtypes: bool(1), int64(7), object(3)
memory usage: 63.4+ KB


### Задачи

<div class="alert alert-info">
<b>Задание № 1:</b>
    
Профессор Оук подозревает, что покемоны в классе `Grass` имеют более сильную обычную атаку, чем покемоны в классе `Rock`. Проверьте, прав ли он, и убедите его в своём выводе статистически.
    
    
Примечание: если есть покемоны, которые относятся к обоим классам, просто выбросьте их;
    
Вы можете предположить, что распределение обычных атак является нормальным для всех классов покемонов.

</div>

In [ ]:
rock_grass = pokemon[(pokemon['Class 1'] == 'Rock') & (pokemon['Class 2'] == 'Grass')] #проверил , что покемонов grass - rock нет, поэтому не учитываем этот порядок классов
rock_grass

,pid,Name,Class 1,Class 2,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Legendary
377,378,Lileep,Rock,Grass,66,41,77,61,87,23,False
378,379,Cradily,Rock,Grass,86,81,97,81,107,43,False


In [ ]:
# Фильтруем траавяных и каменных покемонов
grass = pokemon.loc[(pokemon['Class 1'] == 'Grass') | (pokemon['Class 2'] == 'Grass')]
rock =  pokemon.loc[(pokemon['Class 1'] == 'Rock') | (pokemon['Class 2'] == 'Rock')]
rock_grass = pokemon[(pokemon['Class 1'] == 'Rock') & (pokemon['Class 2'] == 'Grass')]

# Удаляем лишних
grass = grass.loc[~grass['pid'].isin(rock_grass['pid'])]
rock = rock.loc[~rock['pid'].isin(rock_grass['pid'])]

In [ ]:
print(f'Средняя атака травяных покемонов: {grass.Attack.mean()}')
print(f'Средняя атака каменных покемонов: {rock.Attack.mean()}')

Средняя атака травяных покемонов: 73.73118279569893
Средняя атака каменных покемонов: 91.78571428571429


In [ ]:
# Проверим равенство дисперсий
stat, p = st.levene(grass['Attack'], rock['Attack'])

if p < 0.05:
  print('Есть основания отвергнуть нулевую гипотезу о равенстве дисперсий')
else:
  print('Нет оснований отвергнуть нулевую гипотезу о равенстве диспресий')

Есть основания отвергнуть нулевую гипотезу о равенстве дисперсий


In [ ]:
stat, p = st.ttest_ind(grass['Attack'], rock['Attack'], equal_var=False) #используем поправку на разные дисперсии

if p < 0.05:
  print('Есть основания отвергнуть нулевую гипотезу о равенстве средних. Каменные покемоны в среднем действительно сильнее')
else:
  print('Нет оснований отвергнуть нулевую гипотезу о равенстве средних. Различия в размере силы атаки не являются стат значимыми')

Есть основания отвергнуть нулевую гипотезу о равенстве средних. Каменные покемоны в среднем действительно сильнее


<div class="alert alert-info">
<b>Задание № 2:</b>
    
Профессор Оук уже долго не может спать по ночам, ведь его волнует вопрос, а правда ли, что покемоны в классе `Water` в среднем быстрее, чем покемоны в классе `Normal`.
    
    
Проверьте, прав ли он, и убедите его в своём выводе статистически.
    
Примечание: если есть покемоны, которые относятся к обоим классам, выбросьте их;
    
Вы можете предположить, что распределение скорости движения является нормальным для всех классов покемонов.
</div>

In [ ]:
# Фильтруем покемонов нужных классов
normal = pokemon.loc[(pokemon['Class 1'] == 'Normal') | (pokemon['Class 2'] == 'Normal')]
water = pokemon.loc[(pokemon['Class 1'] == 'Water') | (pokemon['Class 2'] == 'Water')]
normal_water = pokemon.loc[((pokemon['Class 2'] == 'Water') & (pokemon['Class 1'] == 'Normal')) |
                           ((pokemon['Class 1'] == 'Water') & (pokemon['Class 2'] == 'Normal'))]

# Удаляем лишних
normal = normal.loc[~normal['pid'].isin(normal_water['pid'])]
water = water.loc[~water['pid'].isin(normal_water['pid'])]

In [ ]:
print(f'Средняя скорость водяных покемонов: {water.Speed.mean()}')
print(f'Средняя скорость нормальных покемонов: {normal.Speed.mean()}')

Средняя скорость водяных покемонов: 64.936
Средняя скорость нормальных покемонов: 72.25742574257426


In [ ]:
# Проверим равенство дисперсий
stat, p = st.levene(water['Speed'], normal['Speed'])

if p < 0.05:
  print('Есть основания отвергнуть нулевую гипотезу о равенстве дисперсий')
else:
  print('Нет оснований отвергнуть нулевую гипотезу о равенстве диспресий')

Есть основания отвергнуть нулевую гипотезу о равенстве дисперсий


In [ ]:
stat, p = st.ttest_ind(water['Speed'], normal['Speed'], equal_var=False) #используем поправку на разные дисперсии

if p < 0.05:
  print('Есть основания отвергнуть нулевую гипотезу о равенстве средних. Нормальные покемоны в среднем быстрее. Профессор ошибся')
else:
  print('Нет оснований отвергнуть нулевую гипотезу о равенстве средних. Различия в скорости не являются стат значимыми')

Есть основания отвергнуть нулевую гипотезу о равенстве средних. Нормальные покемоны в среднем быстрее. Профессор ошибся


<div class="alert alert-info">
<b>Задание № 3:</b>
    
Профессор Оук тот еще безумец. Он изобрёл сыворотку, способную ускорить покемона. Однако мы усомнились в эффективности его вакцины. Професоор дал эту сыворотку следующим покемонам: смотри массив `treathed_pokemon`. Проверьте, работает ли вообще его сыворотка, убедите всех в своём выводе статистически.
    
    
Вы можете предположить, что распределение скорости движения является нормальным для всех классов покемонов.

</div>

In [ ]:
# Покемоны, которые принимали сыворотку увеличения скорости
treathed_pokemon = ['Mega Beedrill', 'Mega Alakazam',
                    'Deoxys Normal Forme', 'Mega Lopunny']

In [ ]:
bustuded_pokemon = pokemon[pokemon['Name'].isin(treathed_pokemon)]


,Speed
19,145
71,150
428,150
476,135


In [ ]:
print(f'Средняя скорость покемонов, которые принимали сыворотку: {bustuded_pokemon.Speed.mean()}')
print(f'Средняя скорость остальных покемонов: { pokemon[~pokemon.Name.isin(treathed_pokemon)].Speed.mean()}')

Средняя скорость покемонов, которые принимали сыворотку: 145.0
Средняя скорость остальных покемонов: 67.89195979899498


In [ ]:
# Проверим равенство дисперсий
stat, p = st.levene(bustuded_pokemon['Speed'],  pokemon[~pokemon['Name'].isin(treathed_pokemon)]['Speed'])

if p < 0.05:
  print('Есть основания отвергнуть нулевую гипотезу о равенстве дисперсий')
else:
  print('Нет оснований отвергнуть нулевую гипотезу о равенстве диспресий')

Есть основания отвергнуть нулевую гипотезу о равенстве дисперсий


In [ ]:
stat, p = st.ttest_ind(bustuded_pokemon['Speed'],  pokemon[~pokemon['Name'].isin(treathed_pokemon)]['Speed'], equal_var=False) #используем поправку на разные дисперсии

if p < 0.05:
  print('Есть основания отвергнуть нулевую гипотезу о равенстве средних. Сыворотка работает')
else:
  print('Нет оснований отвергнуть нулевую гипотезу о равенстве средних. Сыворотка не работает')

Есть основания отвергнуть нулевую гипотезу о равенстве средних. Нормальные покемоны в среднем быстрее. Профессор ошибся


<div class="alert alert-info">
<b>Задание № 4:</b>
    
Профессор Оук всегда любил истории про легендарных покемонов. Однако профессор не очень уверен, что они лучше остальных покемонов. Оук предложил разобраться в этом нам. Проверьте, действительно ли сумма характеристик `HP`,`Attack`,`Defense` у легендарных покемонов выше, чем у других покемонов?

А произведение этих же параметров?

Найдите ответы на эти вопросы и убедите всех в своём выводе статистически.
   

Вы можете предположить, что распределение сум и произведений этих параметров является нормальным для всех классов покемонов.

</div>

In [ ]:
# фильтруем покемонов
legendary = pokemon[pokemon['Legendary'] == True]
common = pokemon[pokemon['Legendary'] == False]

# Создаем столбец с суммой характеристик
legendary['sum_of_3'] = legendary.HP + legendary.Attack +  legendary.Defense
common['sum_of_3'] = common.HP + common.Attack +  common.Defense

# Создаем столбец с произведением характеристик
legendary['mult_of_3'] = legendary.HP * legendary.Attack *  legendary.Defense
common['mult_of_3'] = common.HP * common.Attack *  common.Defense

Сумма характеристик

In [ ]:
print(f'Среднее значение суммы основных характеристик легендарных покемонов: {legendary.sum_of_3.mean()}')
print(f'Среднее значение суммы основных характеристик обычных покемонов: {common.sum_of_3.mean()}')

Среднее значение суммы основных характеристик легендарных покемонов: 309.0769230769231
Среднее значение суммы основных характеристик обычных покемонов: 214.4108843537415


In [ ]:
# Проверим равенство дисперсий
stat, p = st.levene(legendary['sum_of_3'], common['sum_of_3'])

if p < 0.05:
  print('Есть основания отвергнуть нулевую гипотезу о равенстве дисперсий')
else:
  print('Нет оснований отвергнуть нулевую гипотезу о равенстве диспресий')

Есть основания отвергнуть нулевую гипотезу о равенстве дисперсий


In [ ]:
stat, p = st.ttest_ind(legendary['sum_of_3'], common['sum_of_3'], equal_var=False) #используем поправку на разные дисперсии

if p < 0.05:
  print('Есть основания отвергнуть нулевую гипотезу о равенстве средних. У легендарных покемонов сумма характеристик выше, чем у других покемонов')
else:
  print('Нет оснований отвергнуть нулевую гипотезу о равенстве средних. Легендарные покемоны не отличаются от обычных')

Есть основания отвергнуть нулевую гипотезу о равенстве средних. У легендарных покемонов сумма характеристик выше, чем у других покемонов


Произведение характеристик


In [ ]:
print(f'Среднее значение произведения основных характеристик легендарных покемонов: {legendary.mult_of_3.mean()}')
print(f'Среднее значение произведения основных характеристик обычных покемонов: {common.mult_of_3.mean()}')

Среднее значение произведения основных характеристик легендарных покемонов: 1085941.6153846155
Среднее значение произведения основных характеристик обычных покемонов: 425041.38911564625


In [ ]:
# Проверим равенство дисперсий
stat, p = st.levene(legendary['mult_of_3'], common['mult_of_3'])

if p < 0.05:
  print('Есть основания отвергнуть нулевую гипотезу о равенстве дисперсий')
else:
  print('Нет оснований отвергнуть нулевую гипотезу о равенстве диспресий')

Есть основания отвергнуть нулевую гипотезу о равенстве дисперсий


In [ ]:
stat, p = st.ttest_ind(legendary['mult_of_3'], common['mult_of_3'], equal_var=False) #используем поправку на разные дисперсии

if p < 0.05:
  print('Есть основания отвергнуть нулевую гипотезу о равенстве средних. У легендарных покемонов произведение характеристик выше, чем у других покемонов')
else:
  print('Нет оснований отвергнуть нулевую гипотезу о равенстве средних. Легендарные покемоны не отличаются от обычных')

Есть основания отвергнуть нулевую гипотезу о равенстве средних. У легендарных покемонов произведение характеристик выше, чем у других покемонов


<div class="alert alert-info">
<b>Задание № 5:</b>
    
Профессор Оук частенько наблюдает за боями покемонов. После очередных таких боёв Оук выделил четыре класса `best_defence_class`, которые на его взгляд одинаковы по "силе обычной защиты" `Defense`.

Проверьте, действительно ли эти классы покемонов не отличаются по уровню защиты статистически значимо? Всё та же статистика вам в помощь!
   

Вы можете предположить, что распределение параметров защитных характеристик является нормальным для всех классов покемонов.

</div>

In [ ]:
best_defence_class = ['Rock', 'Ground', 'Steel', 'Ice']

# Функция фильтрации
def filter_pokemon(row):
    classes = {row['Class 1'], row['Class 2']} - {None}  # Убираем None
    return len(classes.intersection(best_defence_class)) == 1  # Проверяем, что ровно один класс из целевых

# Применяем фильтр
best_defence_pokemon = pokemon[pokemon.apply(filter_pokemon, axis=1)]

rock = best_defence_pokemon[(best_defence_pokemon['Class 1'] == 'Rock')|(best_defence_pokemon['Class 2'] == 'Rock')]
ground = best_defence_pokemon[(best_defence_pokemon['Class 1'] == 'Ground')|(best_defence_pokemon['Class 2'] == 'Ground')]
steel = best_defence_pokemon[(best_defence_pokemon['Class 1'] == 'Steel')|(best_defence_pokemon['Class 2'] == 'Steel')]
ice = best_defence_pokemon[(best_defence_pokemon['Class 1'] == 'Ice')|(best_defence_pokemon['Class 2'] == 'Ice')]

In [ ]:
print(f'Средняя сила защиты у каменных покемонов: {rock.Defense.mean()}')
print(f'Средняя сила защиты у земляных покемонов: {ground.Defense.mean()}')
print(f'Средняя сила защиты у стальных покемонов: {steel.Defense.mean()}')
print(f'Средняя сила защиты у ледяных покемонов: {ice.Defense.mean()}')

Средняя сила защиты у каменных покемонов: 104.09756097560975
Средняя сила защиты у земляных покемонов: 81.07692307692308
Средняя сила защиты у стальных покемонов: 109.325
Средняя сила защиты у ледяных покемонов: 78.51515151515152


In [ ]:
# так как имеем 4 выборки применим метод ANOVA
stat, p = st.f_oneway(rock.Defense, ground.Defense, steel.Defense, ice.Defense)
alpha = 0.05
if p < alpha:
  print("Отклоняем нулевую гипотезу >> Различие между выборками статистически значимо")
else:
  print("Не отклоняем нулевую гипотезу >> Выборки не имеют статистически значимых различий")

Отклоняем нулевую гипотезу >> Различие между выборками статистически значимо


In [ ]:
# Прменим тест тьюки, чтобы узнать в каких парах есть значимые различия
res = st.tukey_hsd(rock.Defense, ground.Defense, steel.Defense, ice.Defense)
alpha = 0.05
print(res)

Tukey's HSD Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     23.021     0.007     4.758    41.284
 (0 - 2)     -5.227     0.898   -24.661    14.206
 (0 - 3)     25.582     0.008     5.133    46.032
 (1 - 0)    -23.021     0.007   -41.284    -4.758
 (1 - 2)    -28.248     0.001   -46.638    -9.858
 (1 - 3)      2.562     0.986   -16.900    22.023
 (2 - 0)      5.227     0.898   -14.206    24.661
 (2 - 1)     28.248     0.001     9.858    46.638
 (2 - 3)     30.810     0.001    10.246    51.373
 (3 - 0)    -25.582     0.008   -46.032    -5.133
 (3 - 1)     -2.562     0.986   -22.023    16.900
 (3 - 2)    -30.810     0.001   -51.373   -10.246



Значимые различия существуют в парах:
* rock - ground
* rock - ice
* ground - steel
* steel - ice